# Tutorial 5 — Structural LTN

Tutorials 1–4 had one kind of index: a **variable** ranges over a batch of independent individuals, and a formula is evaluated on each of them (or on each combination). Many data have a second kind of index, *inside* one individual: the days of a hospital stay, the frames of a video, the nodes of a graph. sLTN calls these **structural dimensions**.

The difference matters for the logic. Two variables are independent: `Fever(x) & Fever(y)` is evaluated on every pair of patients. The days of a stay are not: "day 3" means day 3 *of the same patient*, days are ordered, and the interesting statements relate days to each other ("the fever persists to the next day", "once recovered, the fever never comes back").

| Concept | Declared with | Role |
|---|---|---|
| structural dimension | `sig.dimension("Day")` | an index set inside each individual; its size comes from the data |
| structural variable | `sig.structural_variable("d", "Day")` | names a position along `Day`; the only way to refer to a dimension in a formula |
| structural relation | `sig.structural_relation("next", ["Day", "Day"])` | a fixed [0, 1] matrix over positions, used as a guard or as an atom |
| consumed / produced dims | `input_dims=`, `output_dims=` | which structural axes a predicate or function receives whole, and which it returns |

The running example is a hospital ward where each patient's temperature is taken once a day. We build the signature and the data, then write formulas about single days, whole stays and pairs of days, and finally train a network that recognises fever days from a few labels and two rules about how a fever evolves.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sltn
from sltn import Signature, Interpretation, KB
from sltn.interpretation.operators.aggregator import AggregPMean, AggregPMeanError, AggregMin, AggregMax

torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1 — Signature: dimensions and structural variables

A **dimension** is declared by name only. It has no size in the signature: the size comes from the tensors grounded over it, and must be the same in all of them.

A dimension never appears in a formula. To refer to its positions, declare a **structural variable** over it: `d` ranges over the days. Writing `forall Day: ...` or `x[Day]` is an error; every structural axis is named by a structural variable.

A variable or constant declared with `dims=["Day"]` holds one value per day: `x` is a batch of patients, each with a reading for every day.

In [ ]:
sig = Signature("ward")
sig.sort("Reading")                          # one temperature reading
sig.dimension("Day")
sig.structural_variable("d", "Day")          # d ranges over the days

sig.variable("x", "Reading", dims=["Day"])   # patients, one reading per day
print(sig)

**Data layout.** `x` is grounded with a tensor `[patients, days, *shape of Reading]`: the variable axis first, then one axis per dimension in the order of `dims`, then the sort's shape. Four patients over five days:

| patient | course |
|---|---|
| 0 | recovers |
| 1 | gets worse |
| 2 | never has fever |
| 3 | recovers, then relapses |

In [ ]:
temps = torch.tensor([
    [39.0, 38.6, 38.0, 37.2, 36.8],   # 0: recovers
    [37.0, 37.4, 38.2, 38.9, 39.3],   # 1: gets worse
    [36.8, 36.9, 36.7, 36.9, 36.8],   # 2: never has fever
    [38.6, 37.1, 36.9, 38.4, 38.8],   # 3: recovers, then relapses
])

interp = Interpretation(sig)
interp["Reading"] = sltn.Type("temperature", shape=(1,))
interp["x"] = temps.unsqueeze(-1)            # [4 patients, 5 days, 1]
print(interp("x[d]"))

`x[d]` is a term with three axes: a **variable** axis `x` (the patients), a **structural** axis `d` (the days) and a **domain** axis `temperature` (the reading).

## 2 — Naming axes

Every structural axis in a formula must be named with a structural variable. `x` alone is rejected, and so is a dimension name:

In [ ]:
sig.predicate("Fever", ["Reading"])
interp["Fever"] = lambda v: torch.sigmoid(3 * (v - 37.8))

for s in ["Fever(x)", "Fever(x[Day])"]:
    try:
        sig.parse(s)
    except (ValueError, NameError) as e:
        print(f"{s}\n  -> {e}\n")

The name is what gives the axis its meaning: **the same name means the same day**. To talk about two days at once, use an alias of `d`: `d1`, `d2`, `d'`, … are created on first use, range over the same dimension and name a new axis.

- `Fever(x[d]) & Fever(x[d])` compares each day with itself: axes `[x, d]`.
- `Fever(x[d]) & Fever(x[d1])` compares every pair of days of the same patient: axes `[x, d, d1]`.

`free_structural_variables` lists the structural axes still free in a parsed formula; it is computed from the syntax, without evaluating anything.

In [ ]:
for s in ["Fever(x[d]) & Fever(x[d])", "Fever(x[d]) & Fever(x[d1])"]:
    phi = sig.parse(s)
    r = interp(phi)
    print(f"{s:30} free structural variables: {phi.free_structural_variables}   axes: {r.axis_names}  shape: {tuple(r.value.shape)}")

The patient axis `x` is shared too: a structural axis indexes positions *inside* each individual, so `d1` is another day of the same patient, never a day of another patient. Pairs of patients would need a second variable (`x1`).

## 3 — Three kinds of predicates

`input_dims` and `output_dims` decide what a predicate's grounding receives and returns. This is the main design choice in a structural signature.

| Declaration | The grounding receives | It returns | In a formula | Use it for |
|---|---|---|---|---|
| no dims | one reading: `[B, 1]` | `[B, 1]` | `Fever(x[d])`: `d` stays free | a property of one day |
| `input_dims=["Day"]` | the whole stay: `[B, Day, 1]` | `[B, 1]` | `Recovered(x[d])`: `d` is **consumed** | a property of the whole stay |
| `input_dims=["Day"], output_dims=["Day"]` | the whole stay: `[B, Day, 1]` | `[B, Day, 1]` | `Rising(x[d])[d]`: the output axis is named | a property of one day that depends on the other days |

- **No dims.** `d` is treated like a variable: sLTN flattens patients and days into one batch, `B = 4 × 5 = 20`, and calls the network on single readings. The network can't see the other days.
- **Consumed.** The network sees each patient's whole sequence (`B = 4`) and returns one value per patient. `d` disappears from the result.
- **Consumed and produced.** The network sees the whole sequence and returns one value per day, e.g. a recurrent or convolutional network. The output axis must be named after the call: `Rising(x[d])[d]`.

`Rising` compares each reading with the previous day; on day 0 there is no previous day, so it compares the reading with itself (0.5). The groundings below print the shape they receive.

In [ ]:
sig.predicate("Recovered", ["Reading"], input_dims=["Day"])
sig.predicate("Rising", ["Reading"], input_dims=["Day"], output_dims=["Day"])


def fever(v):
    print("  Fever receives    ", tuple(v.shape))
    return torch.sigmoid(3 * (v - 37.8))


def recovered(v):                                # v: [B, Day, 1]
    print("  Recovered receives", tuple(v.shape))
    return torch.sigmoid(3 * (37.5 - v[:, -1]))  # last reading below 37.5: [B, 1]


def rising(v):                                   # v: [B, Day, 1]
    print("  Rising receives   ", tuple(v.shape))
    prev = torch.cat([v[:, :1], v[:, :-1]], dim=1)
    return torch.sigmoid(3 * (v - prev))         # [B, Day, 1]


interp["Fever"] = fever
interp["Recovered"] = recovered
interp["Rising"] = rising

for s in ["Fever(x[d])", "Recovered(x[d])", "Rising(x[d])[d]"]:
    print(s)
    r = interp(s)
    print(f"  result axes {r.axis_names}, shape {tuple(r.value.shape)}\n")

`Fever` and `Rising` both give one truth degree per patient and day, but only `Rising` can depend on the other days. Declared like `Rising`, a grounding that looks at one reading at a time would give the same numbers as `Fever`: the choice is about what the network must be able to see.

A predicate that consumes `Day` must be told which axis of its argument to consume, so `Recovered(x)` is an error too. A predicate that produces `Day` must name its output axis: `Rising(x[d])` is rejected.

In [ ]:
interp["Fever"] = lambda v: torch.sigmoid(3 * (v - 37.8))     # quiet again
interp["Recovered"] = lambda v: torch.sigmoid(3 * (37.5 - v[:, -1]))
interp["Rising"] = lambda v: torch.sigmoid(3 * (v - torch.cat([v[:, :1], v[:, :-1]], dim=1)))

try:
    sig.parse("Rising(x[d])")
except (ValueError, NameError) as e:
    print("Rising(x[d])\n  ->", e)

**An argument without the consumed axis.** A predicate can take other arguments next to the sequence. Say we want to compare a patient's stay with a target temperature. The target is one reading, with no days, so it is a plain constant:

In [ ]:
sig.constant("target", "Reading")
sig.predicate("BelowOnAverage", ["Reading", "Reading"], input_dims=["Day"])

interp["target"] = torch.tensor([37.5])
interp["BelowOnAverage"] = lambda v, c: torch.sigmoid(3 * (c - v.mean(dim=1)))

try:
    interp("BelowOnAverage(x[d], target)")
except RuntimeError as e:
    print(e)

The grounding looks right: `v.mean(dim=1)` averages the days, giving `[4, 1]`, and the target should be `[4, 1]` too. But the error shows that it received `(4, 5, 1)`, even though `target` has no days.

This is the rule: **every argument receives all the consumed axes of the call**, also an argument that doesn't have them. `Day` is consumed, so `target` arrives as `[B, Day, 1]`, with the same value copied on every day. The grounding averaged the days of `v` first, so `Day` was gone when it met `target`, and the shapes no longer matched.

The error says all of this: which symbol failed, the shapes it received, which argument has no `d`, and where in its shape the copies are. The copies are identical, so keeping the first one loses nothing. `c.select(1, 0)` takes index 0 along position 1:

In [ ]:
def below_on_average(v, c):                     # v: [B, Day, 1], c: [B, Day, 1]
    print("  BelowOnAverage receives", tuple(v.shape), tuple(c.shape))
    return torch.sigmoid(3 * (c.select(1, 0) - v.mean(dim=1)))   # [B, 1]


interp["BelowOnAverage"] = below_on_average
print(interp("BelowOnAverage(x[d], target)").value.squeeze(-1))

Why copy at all? Because another grounding wants exactly that. Here we compare each day with the target and then average the truth degrees. The target is used day by day, so the copies line up with the readings and no extra line is needed:

In [ ]:
sig.predicate("BelowMostDays", ["Reading", "Reading"], input_dims=["Day"])


def below_most_days(v, c):                      # v: [B, Day, 1], c: [B, Day, 1]
    print("  BelowMostDays receives ", tuple(v.shape), tuple(c.shape))
    return torch.sigmoid(3 * (c - v)).mean(dim=1)   # one degree per day, then [B, 1]


interp["BelowMostDays"] = below_most_days
print(interp("BelowMostDays(x[d], target)").value.squeeze(-1))

The two predicates have the same declaration and receive the same shapes: only the inside of the grounding differs, and `input_dims` can't tell them apart.

| The grounding... | Example | The argument without `Day` |
|---|---|---|
| removes `Day` first, then combines | average, or a GRU that returns one vector per sequence | drop the copies: `c.select(1, 0)` |
| combines day by day, then removes `Day` | a comparison per day, or attention over the days | use it as it is |

Always copying gives one rule. A mistake is usually an error, but not always: if the batch happens to have as many rows as there are days, a grounding can get the right shape by the wrong route, so test with sizes that differ.

**Renaming the output axis.** The name after the call is chosen freely. `Rising(x[d])[d]` keeps the input's name, so the output day is matched with the same day elsewhere in the formula. A new name, `Rising(x[d])[d1]`, puts the output on its own axis:

In [ ]:
for s in ["Fever(x[d]) & Rising(x[d])[d]", "Fever(x[d]) & Rising(x[d])[d1]"]:
    r = interp(s)
    print(f"{s:35} axes {r.axis_names}")

The first reads "fever on day `d` and rising on day `d`"; the second, "fever on day `d` and rising on day `d1`", for every pair.

**Functions** take the same `input_dims` / `output_dims`. A smoothing function returns a sequence of the same length (consumes and produces `Day`); the mean temperature of the stay consumes `Day`. Their results are terms, used inside predicates:

In [ ]:
sig.function("smooth", ["Reading"], "Reading", input_dims=["Day"], output_dims=["Day"])
sig.function("mean_temp", ["Reading"], "Reading", input_dims=["Day"])


def moving_average(v):                           # v: [B, Day, 1], 3-day window
    padded = F.pad(v.transpose(1, 2), (1, 1), mode="replicate")
    return F.avg_pool1d(padded, 3, stride=1).transpose(1, 2)


interp["smooth"] = moving_average
interp["mean_temp"] = lambda v: v.mean(dim=1)    # [B, 1]

print(interp("smooth(x[d])[d]").axis_names)
print(interp("Fever(smooth(x[d])[d])").value.squeeze(-1))
print(interp("Fever(mean_temp(x[d]))").value.squeeze(-1))

## 4 — Quantifying over days

`∀ d: φ` and `∃ d: φ` aggregate the axis `d` away. They are ordinary `Forall` / `Exists` nodes; sLTN sees that `d` is a structural variable and aggregates the `d` axis.

**The operators must be set for each dimension.** The `Logic`'s `∀` and `∃` apply to variables only; a structural quantifier uses the operator registered for its dimension, and there is no default:

In [ ]:
try:
    interp("∀ d: Fever(x[d])")
except ValueError as e:
    print(e)

The key is `"<quantifier>,<dimension>"`. This is the one place where the dimension's name is used; every structural variable over `Day` (`d`, `d1`, …) uses this entry. As in Tutorial 2, pMeanError and pMean are smooth versions of min and max, suited to learning.

In [ ]:
interp.logic["∀,Day"] = AggregPMeanError(p=2)
interp.logic["∃,Day"] = AggregPMean(p=2)

always = interp("∀ d: Fever(x[d])").value.squeeze(-1)
some_day = interp("∃ d: Fever(x[d])").value.squeeze(-1)
for k in range(4):
    print(f"patient {k}: fever every day {always[k]:.2f}   fever on some day {some_day[k]:.2f}")

`x` is still free, so there is one value per patient. With min and max the readings are crisper ("every day" is the coldest day, "some day" the hottest), but each value then depends on a single day, and so does its gradient:

In [ ]:
interp.logic["∀,Day"] = AggregMin()
interp.logic["∃,Day"] = AggregMax()
print("min:", interp("∀ d: Fever(x[d])").value.squeeze(-1))
print("max:", interp("∃ d: Fever(x[d])").value.squeeze(-1))

interp.logic["∀,Day"] = AggregPMeanError(p=2)
interp.logic["∃,Day"] = AggregPMean(p=2)

**Closing the formula.** A quantifier binds either individual variables or structural variables, not both: `∀ x, d: ...` is an error. Nest them instead. Reading the formulas:

- `∀ x: ∃ d: Fever(x[d])`: every patient has fever on some day;
- `∃ d: ∀ x: Fever(x[d])`: there is a day on which every patient has fever.

In [ ]:
try:
    sig.parse("∀ x, d: Fever(x[d])")
except (ValueError, NameError) as e:
    print("∀ x, d: ...\n  ->", e, "\n")

for s in ["∀ x: ∃ d: Fever(x[d])", "∃ d: ∀ x: Fever(x[d])"]:
    print(f"{s:25} {interp(s).value.item():.3f}")

## 5 — Structural relations

A **structural relation** is a fixed matrix over positions, grounded with a tensor of values in [0, 1]. Here `next(d, d1)` holds when `d1` is the day after `d`, and `before(d, d1)` when `d1` comes later than `d`. A relation is grounded with a tensor, never with a function.

In [ ]:
sig.structural_relation("next", ["Day", "Day"])
sig.structural_relation("before", ["Day", "Day"])

days = torch.arange(5)
interp["next"] = (days.unsqueeze(1) + 1 == days.unsqueeze(0)).float()   # [5, 5]
interp["before"] = (days.unsqueeze(1) < days.unsqueeze(0)).float()
print(interp["next"])
print(interp["before"])

### 5.1 As a guard

A relation after `|` restricts the quantifier to the positions where it holds, as a guard does in Tutorial 2. "The fever persists to the next day":

```
∀ d, d1 | next(d, d1): (Fever(x[d]) -> Fever(x[d1]))
```

The quantifier only aggregates the 4 pairs of consecutive days; the other 21 pairs are left out. The check below computes the same value with a loop over consecutive days.

In [ ]:
persists = interp("∀ d, d1 | next(d, d1): (Fever(x[d]) -> Fever(x[d1]))").value.squeeze(-1)

f = interp("Fever(x[d])").value.squeeze(-1)                     # [patients, days]
implies = interp.logic["->"]
pairs = torch.stack([implies(f[:, k], f[:, k + 1]) for k in range(4)], dim=1)
by_hand = 1 - ((1 - pairs) ** 2).mean(dim=1).sqrt()            # pMeanError, p = 2

print("guarded:", persists)
print("by hand:", by_hand)

Patients 0 and 3 get the lowest values: their fever stops, so on the day it stops `Fever(x[d]) -> Fever(x[d1])` is false. Patient 1, whose fever only grows, and patient 2, who never has fever, satisfy it.

"Once recovered, the fever never comes back" relates a day to all later ones, through `before`. Patient 3 (relapse) violates it, and so does patient 1, who starts without fever and gets it later:

In [ ]:
no_relapse = interp("∀ d, d1 | before(d, d1): (¬Fever(x[d]) -> ¬Fever(x[d1]))").value.squeeze(-1)
for k in range(4):
    print(f"patient {k}: {no_relapse[k]:.2f}")

### 5.2 Soft weights and unary relations

A relation with values between 0 and 1 weighs the positions instead of keeping or dropping them: the aggregation becomes a weighted mean. A relation can also take one argument. `Recent(d)` gives the later days more weight, so "no fever lately" depends mostly on the end of the stay:

In [ ]:
sig.structural_relation("Recent", ["Day"])
interp["Recent"] = torch.tensor([0.1, 0.2, 0.4, 0.7, 1.0])

lately = interp("∀ d | Recent(d): ¬Fever(x[d])").value.squeeze(-1)
uniform = interp("∀ d: ¬Fever(x[d])").value.squeeze(-1)
for k in range(4):
    print(f"patient {k}: no fever lately {lately[k]:.2f}   no fever ever {uniform[k]:.2f}")

To read the numbers: with a guard, the quantifier is a **weighted** pMeanError, $1 - \sqrt{\sum_d w_d (1 - \varphi_d)^2 / \sum_d w_d}$, with $\varphi_d$ = `¬Fever` on day $d$ and $w_d$ = `Recent(d)`. Each day counts in proportion to its weight: the last two days carry 71% of the total weight (0.7 + 1.0 out of 2.4), the first two only 12%. The `Fever` values per day are:

| patient | day 0 | day 1 | day 2 | day 3 | day 4 |
|---|---|---|---|---|---|
| 0 (recovers) | 0.97 | 0.92 | 0.65 | 0.14 | 0.05 |
| 1 (gets worse) | 0.08 | 0.23 | 0.77 | 0.96 | 0.99 |
| 2 (never) | 0.05 | 0.06 | 0.04 | 0.06 | 0.05 |
| 3 (relapses) | 0.92 | 0.11 | 0.06 | 0.86 | 0.95 |

- **Patient 0** had fever on days 0–1 and none at the end. With equal weights, the two fever days pull "no fever ever" down to 0.33; with `Recent` they count little, and "no fever lately" rises to 0.57. Day 2 (0.65, weight 0.4) keeps it from going higher.
- **Patient 1** is the opposite: the fever is on the last days, where the weights are largest, so "lately" (0.12) is lower than "ever" (0.28).
- **Patient 2** never has fever: every day agrees, so the weights change nothing (0.95 in both columns).
- **Patient 3** has fever on day 0 and again on days 3–4. The relapse falls on the heavy days, so "lately" (0.21) is lower than "ever" (0.29).

The weights change how much each day counts, not how true it is: `¬Fever` has the same values in both columns.

### 5.3 As an atom

A relation is also a formula with its own axes, so it can be used in the body. The result is different from the guard:

In [ ]:
guard = interp("∀ d, d1 | next(d, d1): (Fever(x[d]) -> Fever(x[d1]))").value.squeeze(-1)
atom = interp("∀ d, d1: (next(d, d1) -> (Fever(x[d]) -> Fever(x[d1])))").value.squeeze(-1)
print("as a guard:", guard)
print("as an atom:", atom)

With the atom, the quantifier still aggregates all 25 pairs of days, and the 21 pairs that are not consecutive count as true (`0 -> …` is 1). They dilute the 4 pairs that matter and pull every value toward 1: patient 0, whose fever stops, goes from 0.67 to 0.87. So a guard and an atom do different jobs. A guard says which pairs the quantifier looks at: the relation decides what the statement is *about*. An atom is part of what the statement *says*: its truth value is combined with the others by the connectives. Use the atom when the relation is part of the claim, for example as a conclusion, which a guard can't express. "Every fever day comes before every day without fever":

In [ ]:
before_fever = interp("∀ d, d1: ((Fever(x[d]) & ¬Fever(x[d1])) -> before(d, d1))").value.squeeze(-1)
for k in range(4):
    print(f"patient {k}: {before_fever[k]:.2f}")

Here `before` is the claim, not a filter:

- **Patient 0** has fever first and none afterwards: true (0.94).
- **Patient 1** has its days without fever *before* the fever days: violated (0.61).
- **Patient 2** has no fever day, so the premise is false on every pair and the formula holds vacuously (0.96).
- **Patient 3** relapses: its fever days on days 3–4 come *after* days without fever (0.66).

**Known limitation:** a relation can't be applied twice to the same variable yet: `next(d, d)` raises `Axis names must be unique`, while `Fever(x[d]) & Fever(x[d])` works. For `next` this costs nothing, since no day is next to itself. It matters for relations whose diagonal means something: on a graph of people, "nobody is their own friend" is about `friends(p, p)`. Until it is supported, declare a separate relation grounded with the identity matrix, `same(p, p1)`, and write `∀ p, p1 | same(p, p1): ¬friends(p, p1)`.

## 6 — Axis selection: picking one day

An **axis selection** `[d=k]` keeps position `k` of the structural axis `d` and removes the axis. Negative indices count from the end, as in Python. The axis selection goes after the atom that names the axis.

In [ ]:
admission = interp("Fever(x[d])[d=0]")
discharge = interp("Fever(x[d])[d=-1]")
print("axes:", admission.axis_names)
print("fever on admission:", admission.value.squeeze(-1))
print("fever at discharge:", discharge.value.squeeze(-1))

A formula with an axis selection combines with the others like any formula. "Admitted with fever, and recovered":

In [ ]:
print(interp("Fever(x[d])[d=0] & Recovered(x[d])").value.squeeze(-1))

try:
    interp("Fever(x[d])[d=7]")                   # only 5 days
except IndexError as e:
    print("\nFever(x[d])[d=7]\n  ->", e)

## 7 — Several dimensions

A symbol can have several dimensions. Here the temperature is taken three times a day: `xh` has one reading per day and per hour, and the annotation names both axes in the order of `dims`.

In [ ]:
sig.dimension("Hour")
sig.structural_variable("h", "Hour")
sig.variable("xh", "Reading", dims=["Day", "Hour"])

interp["xh"] = temps.unsqueeze(-1).unsqueeze(-1) + torch.tensor([-0.4, 0.0, 0.4]).view(1, 1, 3, 1)  # [4, 5, 3, 1]
interp.logic["∃,Hour"] = AggregMax()

r = interp("Fever(xh[d, h])")
print("Fever(xh[d, h]) axes:", r.axis_names, tuple(r.value.shape))
print("fever at some hour, every day:", interp("∀ d: ∃ h: Fever(xh[d, h])").value.squeeze(-1))

- A quantifier binds structural variables of **one** dimension: `∀ d, h: ...` is an error; nest `∀ d: ∃ h: ...`.
- **Read results by axis name, not by position.** Before combining tensors, sLTN puts their axes in a canonical order: variable axes in alphabetical order, then structural axes in alphabetical order, then the domain axes. A result's axis order can therefore differ from the order of `dims` (with structural variables named `t` and `a`, a result comes out `[..., a, t, ...]`). `r.axis_names` gives the order.
- The annotation follows the order of `dims`: `xh[h, d]` is rejected, because the first axis of `xh` is a `Day`.

In [ ]:
try:
    sig.parse("Fever(xh[h, d])")
except (ValueError, NameError) as e:
    print(e)

## 8 — Learning with structure

A ward of 200 patients observed for 7 days. Every patient is admitted with fever, which lasts 1 to 5 days and does not come back. Each day the patient's temperature is measured once. On a day with fever, the reading is drawn around 38.1 °C; on a day without, around 36.9 °C; both with a standard deviation of 0.5 °C, so the two overlap and a single reading is often ambiguous. A doctor labelled only **5%** of the days (never the admission day). We train `Fever` from these labels, with and without what we know about the course of the disease.

**Patients as a dimension.** A label is about one reading: "patient 17 had fever on day 3" must be compared with `temp[17, 3]` and nothing else. The labelled days are given as masks over (patient, day), and a structural relation can only range over dimensions, not over individual variables. So here the patients are a second dimension, `Patient`, and the readings are a **constant** with dims `[Patient, Day]`: one tensor `[200, 7, 1]`. In `Fever(temp[p, d])[d]` guarded by `Sick(p, d)`, the reading and the mask share `p` and `d`, so each label meets exactly its own reading.

With patients as an individual variable instead, the labels would have to be carried by a second variable, and two variables are crossed (Tutorial 1, 3.2): every patient's reading would meet every patient's label. Diagonal quantification, `∀ (x, y)` (Tutorial 2, section 3), pairs them one to one, and is the usual way in LTN to keep data and labels together.

**Labels as masks.** The labelled days are two relations over `[Patient, Day]`, `Sick` and `Healthy`, used as guards: each clause reads as a sentence ("on every day labelled sick, `Fever` holds"), and the unlabelled days drop out of the quantifier (Section 5.1).

In [ ]:
torch.manual_seed(0)
N, T = 200, 7
last = torch.randint(1, 6, (N,))                                         # fever lasts 1..5 days
fever_true = (torch.arange(T).unsqueeze(0) < last.unsqueeze(1)).float()  # [N, T]
readings = 36.9 + 1.2 * fever_true + 0.5 * torch.randn(N, T)
labelled = torch.rand(N, T) < 0.05
labelled[:, 0] = False                                                   # admission day never labelled
print(f"{labelled.sum().item()} labelled days out of {N * T}")
print(f"accuracy of the threshold 37.5 on single readings: "
      f"{((readings > 37.5).float() == fever_true)[~labelled].float().mean():.3f}")

`Fever` consumes and produces `Day` (Section 3): a bidirectional GRU that reads the whole stay and returns one truth degree per day, so it can use the neighbouring days to decide an ambiguous reading. `p` is not consumed, so the 200 patients form the batch.

In [ ]:
ward = Signature("ward_learning")
ward.sort("Reading")
ward.dimension("Patient")
ward.structural_variable("p", "Patient")
ward.dimension("Day")
ward.structural_variable("d", "Day")

ward.constant("temp", "Reading", dims=["Patient", "Day"])
ward.predicate("Fever", ["Reading"], input_dims=["Day"], output_dims=["Day"])
ward.structural_relation("Sick", ["Patient", "Day"])
ward.structural_relation("Healthy", ["Patient", "Day"])
ward.structural_relation("next", ["Day", "Day"])


class FeverNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.gru = nn.GRU(1, 16, batch_first=True, bidirectional=True)
        self.out = nn.Linear(32, 1)

    def forward(self, v):                    # v: [B, Day, 1]
        h, _ = self.gru(v - 37.5)
        return torch.sigmoid(self.out(h))    # [B, Day, 1]


def make_interp():
    i = Interpretation(ward)
    i.to(device)
    i["Reading"] = sltn.Type("temperature", shape=(1,))
    i["temp"] = readings.unsqueeze(-1)
    i["Sick"] = (labelled & (fever_true == 1)).float()
    i["Healthy"] = (labelled & (fever_true == 0)).float()
    days = torch.arange(T)
    i["next"] = (days.unsqueeze(1) + 1 == days.unsqueeze(0)).float()
    i["Fever"] = FeverNet()
    i.logic["∀,Patient"] = AggregPMeanError(p=4)
    i.logic["∀,Day"] = AggregPMeanError(p=4)
    return i

The knowledge base. The first two clauses are the labels. The other two are what we know:

- every patient is admitted with fever: `Fever` on day 0, an axis selection;
- the fever doesn't come back: if there is fever on the next day, there was fever on this day, a guard with `next`.

Both dimensions use pMeanError with p = 4, closer to the minimum than the p = 2 of the previous sections: a clause is satisfied only if it holds for (almost) every patient and day, not just on average. With p = 2 the two rules were too weak, and on our runs the gain from them came and went with the seed.

In [ ]:
labels = [
    ward.parse("∀ p: ∀ d | Sick(p, d): Fever(temp[p, d])[d]"),
    ward.parse("∀ p: ∀ d | Healthy(p, d): ¬Fever(temp[p, d])[d]"),
]
knowledge = [
    ward.parse("∀ p: Fever(temp[p, d])[d][d=0]"),
    ward.parse("∀ p: ∀ d, d1 | next(d, d1): (Fever(temp[p, d1])[d1] -> Fever(temp[p, d])[d])"),
]


def train(clauses, epochs=300):
    torch.manual_seed(100)
    i = make_interp()
    kb = KB(clauses)
    optimizer = torch.optim.Adam(i.parameters(), lr=1e-2)
    for _ in range(epochs):
        optimizer.zero_grad()
        loss = 1 - i(kb).value.squeeze()
        loss.backward()
        optimizer.step()
    return i


def accuracy_unlabelled(i):
    with torch.no_grad():
        r = i("Fever(temp[p, d])[d]")
    assert r.axis_names[:2] == ["p", "d"]
    predicted = (r.value.squeeze(-1).cpu() > 0.5).float()
    return (predicted == fever_true)[~labelled].float().mean().item()


only_labels = train(labels)
with_knowledge = train(labels + knowledge)
print(f"accuracy on the unlabelled days, labels only:      {accuracy_unlabelled(only_labels):.3f}")
print(f"accuracy on the unlabelled days, labels + knowledge: {accuracy_unlabelled(with_knowledge):.3f}")

Both networks beat the threshold on single readings, because they see the whole stay. The two clauses about the course of the disease add up to 4 points on top: on three random seeds, 0.92 / 0.90 / 0.87 with the labels only and 0.93 / 0.92 / 0.91 with the knowledge. They need no labels: they hold for every patient, so they also constrain the 95% of days nobody labelled.

## Summary

- A **dimension** is an index set inside each individual (days, frames, nodes). Its size comes from the data and must agree across every tensor over it.
- A dimension is referred to only through a **structural variable** (`d` over `Day`). Every structural axis is named: `x[d]`, never `x` or `x[Day]`. The same name means the same position; an alias (`d1`) names a second one.
- `input_dims` / `output_dims` decide what a grounding sees: no dims = one position at a time; consumed = the whole sequence, one value back; consumed and produced = the whole sequence, one value per position (`Rising(x[d])[d]`). Every argument receives the consumed axes, as copies if it doesn't have them; a grounding that removes the axis first drops them with `select(1, 0)`.
- `∀ d` / `∃ d` need an operator for the dimension: `logic["∀,Day"]`, `logic["∃,Day"]`. A quantifier binds variables of one kind and one dimension; nest the others.
- A **structural relation** is a fixed [0, 1] tensor. As a guard it restricts (0/1) or weighs (soft values) the positions a quantifier aggregates; as an atom it is one more condition, and doesn't restrict anything.
- An axis selection `[d=k]` keeps one position of a structural axis and removes the axis; negative indices count from the end.
- Read results by axis name (`r.axis_names`): their axes are in alphabetical order within each role.
- A structural relation ranges over dimensions only: to mask (patient, day) pairs, the patients must be a dimension. Data and labels carried by two variables are paired with `∀ (x, y)` (Tutorial 2), not crossed.

**Common mistakes:**

- forgetting `logic["∀,<Dimension>"]` before quantifying over a dimension;
- declaring a per-position predicate without dims when the network needs the other positions, or the reverse;
- using a relation in the body when a guard was meant;
- grounding tensors over the same dimension with different sizes.